In [ ]:
# test_milvus_vector_store_class.py

from milvus_client import MilvusVectorStore
from models import ChunkRecord
from settings import get_settings
import random

settings = get_settings()


class MilvusVectorStoreTesting:
    """Class to test MilvusVectorStore functionality."""

    def __init__(self):
        print("Initializing MilvusVectorStore for testing...")
        self.store = MilvusVectorStore()
        self.embeddings = []

    @staticmethod
    def generate_fake_embedding(dim: int = settings.embedding_dim) -> list[float]:
        """Generate a random embedding vector of given dimension."""
        return [random.random() for _ in range(dim)]

    def test_insert(self):
        print("---- Testing insert() ----")

        chunks = [
            ChunkRecord(
                text="Machine learning is amazing",
                doc_id="doc1",
                file_name="file1.txt",
                page_number=1,
                chunk_id="chunk1",
                chunk_type="text",
            ),
            ChunkRecord(
                text="Cooking pasta recipe",
                doc_id="doc2",
                file_name="file2.txt",
                page_number=2,
                chunk_id="chunk2",
                chunk_type="text",
            ),
        ]

        self.embeddings = [
            self.generate_fake_embedding(),
            self.generate_fake_embedding(),
        ]

        self.store.insert(chunks, self.embeddings)
        print("Insert successful!\n")

    def test_search(self):
        if not self.embeddings:
            raise ValueError("No embeddings available. Run test_insert first.")

        print("---- Testing search() ----")
        results = self.store.search(self.embeddings[0], top_k=2)

        for r in results:
            print(f"ID: {r.id}")
            print(f"Text: {r.text}")
            print(f"Score: {r.score}")
            print("------")

        print("Search successful!\n")
        return results

    def test_keyword_filter(self, results):
        print("---- Testing keyword_filter() ----")
        query = "machine learning"
        reranked = self.store.keyword_filter(results, query)

        for r in reranked:
            print(f"Reranked ID: {r.id}")
            print(f"Text: {r.text}")
            print(f"Score: {r.score}")
            print("------")

        print("Keyword filter successful!\n")

    def run_all_tests(self):
        print("RUNNING ALL TESTS 🚀")
        self.test_insert()
        search_results = self.test_search()
        self.test_keyword_filter(search_results)
        print("ALL TESTS COMPLETED SUCCESSFULLY ✅")





tester = MilvusVectorStoreTesting()
tester.run_all_tests()

Initializing MilvusVectorStore for testing...
RUNNING ALL TESTS 🚀
---- Testing insert() ----
Insert successful!

---- Testing search() ----
ID: 464410574783775095
Text: Machine learning is amazing
Score: 1.0000001192092896
------
ID: 464410574783775092
Text: Machine learning is amazing
Score: 0.7506145238876343
------
Search successful!

---- Testing keyword_filter() ----
Reranked ID: 464410574783775095
Text: Machine learning is amazing
Score: 1.0000001192092896
------
Reranked ID: 464410574783775092
Text: Machine learning is amazing
Score: 0.7506145238876343
------
Keyword filter successful!

ALL TESTS COMPLETED SUCCESSFULLY ✅


In [5]:
import unittest
from unittest.mock import patch, Mock
import requests

from ollama_client import OllamaClient
from exceptions import OllamaTimeoutError


class TestOllamaClient(unittest.TestCase):

    def setUp(self):
        class MockSettings:
            ollama_base_url = "http://localhost:11434"
            request_timeout_seconds = 5
            embedding_model = "test-embed-model"
            llm_model = "test-llm-model"

        patcher = patch("ollama_client.settings.get_settings", return_value=MockSettings())
        self.addCleanup(patcher.stop)
        self.mock_settings = patcher.start()

        self.client = OllamaClient()

    # -----------------------
    # EMBED TESTS
    # -----------------------

    @patch("ollama_client.requests.post")
    def test_embed_success(self, mock_post):
        mock_response = Mock()
        mock_response.json.return_value = {"embedding": [0.1, 0.2]}
        mock_response.raise_for_status.return_value = None
        mock_post.return_value = mock_response

        result = self.client.embed("hello")

        self.assertEqual(result, [0.1, 0.2])

    @patch("ollama_client.requests.post")
    def test_embed_empty_embedding(self, mock_post):
        mock_response = Mock()
        mock_response.json.return_value = {"embedding": []}
        mock_response.raise_for_status.return_value = None
        mock_post.return_value = mock_response

        with self.assertRaises(OllamaTimeoutError):
            self.client.embed("hello")

    @patch("ollama_client.requests.post", side_effect=requests.Timeout)
    def test_embed_timeout(self, mock_post):
        with self.assertRaises(OllamaTimeoutError):
            self.client.embed("hello")

    # -----------------------
    # GENERATE TESTS
    # -----------------------

    @patch("ollama_client.requests.post")
    def test_generate_success(self, mock_post):
        mock_response = Mock()
        mock_response.json.return_value = {
            "message": {"content": "Hello world"}
        }
        mock_response.raise_for_status.return_value = None
        mock_post.return_value = mock_response

        result = self.client.generate(
            [{"role": "user", "content": "Hi"}]
        )

        self.assertEqual(result, "Hello world")

    @patch("ollama_client.requests.post", side_effect=requests.Timeout)
    def test_generate_timeout(self, mock_post):
        with self.assertRaises(OllamaTimeoutError):
            self.client.generate(
                [{"role": "user", "content": "Hi"}]
            )


unittest.main(argv=["first-arg-is-ignored"], exit=False)

.....
----------------------------------------------------------------------
Ran 5 tests in 0.007s

OK


In [6]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection
from pymilvus import utility

# Connect
connections.connect(host="localhost", port="19530")

# Define fields
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=768),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
]

schema = CollectionSchema(fields, description="Test collection")

collection_name = "test_collection"

# Drop if exists (avoid conflicts)
if utility.has_collection(collection_name=collection_name):
    utility.drop_collection(collection_name)

collection = Collection(name=collection_name, schema=schema)

# Create index
collection.create_index(
    field_name="embedding",
    index_params={
        "index_type": "HNSW",
        "metric_type": "COSINE",
        "params": {"M": 16, "efConstruction": 200},
    },
)

collection.load()

# Insert dummy vector
dummy_vector = [0.1] * 768
text_data = ["hello world"]

collection.insert([[dummy_vector], text_data])

collection.flush()

# Search
results = collection.search(
    data=[dummy_vector],
    anns_field="embedding",
    param={"metric_type": "COSINE", "params": {"ef": 64}},
    limit=1,
    output_fields=["text"]
)

print("Search Result:", results[0][0].entity.get("text"))

Search Result: hello world


In [ ]:
import pandas as pd

df = pd.read_excel("data/sample5.xlsx")
df.sample()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
1280,B07GWTWFS2,KENT 16025 Sandwich Grill 700W | Non-Toxic Cer...,Home&Kitchen|Kitchen&HomeAppliances|SmallKitch...,"â‚¹1,699","â‚¹1,975",0.14,4.1,4716.0,"Compact and sleek looking modern appliance, KE...","AEY6PEMQ7DII44WSUSC67JEWDE3A,AFMVVM2AXA3WHTC23...","Tabassum,Ark,AMIT GROVER,-,deshraj s.,Ankisha ...","RXPUKJKEHY256,R1DXJ48GMFWOZD,R24RXWIR7PL4IW,R1...",Only for grill sandwich use cord length is too...,Good quality but cord is too short .it is ok f...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Kent-16025-700-Watt-Sand...


In [4]:
print(df.loc[1280])

product_id                                                    B07GWTWFS2
product_name           KENT 16025 Sandwich Grill 700W | Non-Toxic Cer...
category               Home&Kitchen|Kitchen&HomeAppliances|SmallKitch...
discounted_price                                                â‚¹1,699
actual_price                                                    â‚¹1,975
discount_percentage                                                 0.14
rating                                                               4.1
rating_count                                                      4716.0
about_product          Compact and sleek looking modern appliance, KE...
user_id                AEY6PEMQ7DII44WSUSC67JEWDE3A,AFMVVM2AXA3WHTC23...
user_name              Tabassum,Ark,AMIT GROVER,-,deshraj s.,Ankisha ...
review_id              RXPUKJKEHY256,R1DXJ48GMFWOZD,R24RXWIR7PL4IW,R1...
review_title           Only for grill sandwich use cord length is too...
review_content         Good quality but cord is too

In [5]:
columns = df.columns

df.sample()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
123,B099K9ZX65,Hisense 108 cm (43 inches) 4K Ultra HD Smart C...,"Electronics|HomeTheater,TV&Video|Televisions|S...","â‚¹20,990","â‚¹44,990",0.53,4.1,1259.0,Resolution: 4K Ultra HD (3840x2160) | Refresh ...,"AFP334GQV3WBH6XJIX5VITMYOH2A,AHAIKXSSOQ7R5GBPV...","Ravi Naresh,anup,Chinmay Bapusaheb Gayake,Heta...","R1Z33CAT0B5EQM,R38KPAP35GXYOK,R26YGSNK20I13P,R...","Hisense Vivid 4K TV Initial Impressions,Pictur...","Hi all, firstly, I was very skeptical to purch...",https://m.media-amazon.com/images/I/51Pu9zNUbt...,https://www.amazon.in/Hisense-inches-Certified...


In [ ]:
from __future__ import annotations

import logging
from typing import Iterable

from pymilvus import Collection, CollectionSchema, DataType, FieldSchema, connections, utility

import settings
from models import ChunkRecord, RetrievalResult
from exceptions import MilvusConnectionError

logger = logging.getLogger("MilvusVectorStore")


class MilvusVectorStore:
    def __init__(self):
        self.settings = settings.get_settings()
        self.collection_name = self.settings.milvus_collection_name
        self._collection: Collection | None = None
        self.connect()
        self.ensure_collection()

    def connect(self) -> None:
        """Establish connection to Milvus."""
        try:
            connections.connect(
                alias="default",
                host=self.settings.milvus_host,
                port=self.settings.milvus_port,
                user=self.settings.milvus_username,
                password=self.settings.milvus_password,
            )
        except Exception as exc:
            raise MilvusConnectionError(f"Failed connecting to Milvus: {exc}") from exc

    def ensure_collection(self) -> None:
        """Ensure the collection exists and is loaded."""
        if not utility.has_collection(self.collection_name):
            fields = [
                FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
                FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=self.settings.embedding_dim),
                FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
                FieldSchema(name="doc_id", dtype=DataType.VARCHAR, max_length=256),
                FieldSchema(name="file_name", dtype=DataType.VARCHAR, max_length=512),
                FieldSchema(name="file_type", dtype=DataType.VARCHAR, max_length=64),    
                FieldSchema(name="page_number", dtype=DataType.INT64),
                FieldSchema(name="chunk_id", dtype=DataType.VARCHAR, max_length=256),
                FieldSchema(name="chunk_type", dtype=DataType.VARCHAR, max_length=64),
            ]
            schema = CollectionSchema(fields=fields, description="Agentic RAG chunks")
            collection = Collection(name=self.collection_name, schema=schema)
            collection.create_index(
                field_name="embedding",
                index_params={
                    "index_type": "HNSW",
                    "metric_type": "COSINE",
                    "params": {"M": 16, "efConstruction": 200},
                },
            )
            logger.info("Created Milvus collection: %s", self.collection_name)

        self._collection = Collection(self.collection_name)
        self._collection.load()

    @property
    def collection(self) -> Collection:
        """Get the collection, ensuring it's initialized."""
        if self._collection is None:
            self.ensure_collection()
        assert self._collection is not None
        return self._collection

    def insert(self, chunks: list[ChunkRecord], embeddings: list[list[float]]) -> None:
        """Insert chunks and embeddings into the collection."""
        if len(chunks) != len(embeddings):
            raise ValueError("Chunks and embeddings length mismatch")
        if not chunks:
            return

        data = [
            embeddings,
            [c.text for c in chunks],
            [c.doc_id for c in chunks],
            [c.file_name for c in chunks],
            [c.file_type for c in chunks],
            [c.page_number for c in chunks],
            [c.chunk_id for c in chunks],
            [c.chunk_type for c in chunks],
        ]

        self.collection.insert(data)
        self.collection.flush()

    def search(self, query_embedding: list[float], top_k: int) -> list[RetrievalResult]:
        """Search the collection for the top_k most similar embeddings."""
        search_params = {"metric_type": "COSINE", "params": {"ef": 128}}
        output_fields = ["text", "doc_id", "file_name", "page_number", "chunk_id", "chunk_type"]

        results = self.collection.search(
            data=[query_embedding],
            anns_field="embedding",
            param=search_params,
            limit=top_k,
            output_fields=output_fields,
        )

        formatted: list[RetrievalResult] = []
        if not results:
            return formatted

        for hit in results[0]:
            e = hit.entity
            formatted.append(
                RetrievalResult(
                    id=int(hit.id),
                    text=getattr(e, "text", ""),
                    doc_id=getattr(e, "doc_id", ""),
                    file_name=getattr(e, "file_name", ""),
                    page_number=int(getattr(e, "page_number", 0)),
                    chunk_id=getattr(e, "chunk_id", ""),
                    chunk_type=getattr(e, "chunk_type", "text"),
                    score=float(hit.score),
                )
            )

        return formatted

    @staticmethod
    def keyword_filter(candidates: Iterable[RetrievalResult], query: str) -> list[RetrievalResult]:
        """Rerank search results based on keyword overlap."""
        query_terms = {t.lower() for t in query.split() if len(t) > 2}
        reranked = []

        for item in candidates:
            text_terms = {t.lower() for t in item.text.split()}
            overlap = len(query_terms.intersection(text_terms))
            hybrid_score = (item.score * 0.8) + (overlap * 0.2)
            reranked.append((hybrid_score, item))

        reranked.sort(key=lambda x: x[0], reverse=True)
        return [item for _, item in reranked]

In [ ]:
from __future__ import annotations
import logging
from pathlib import Path
import hashlib
from typing import List

from llama_index.core import SimpleDirectoryReader, Document

from settings import get_settings
from models import ChunkRecord
from text_preprocess import clean_text, tokenize_words, overlap_chunks
from milvus_client import MilvusVectorStore 
import ollama
import pandas as pd

logger = logging.getLogger("IngestionPipeline")
logging.basicConfig(level=logging.INFO)


class DocumentLoader:
    SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".pptx", ".xlsx", ".xls", ".csv", ".txt", ".md"}

    def load_data(self, file_path: str) -> List[Document]:
        path = Path(file_path)
        if not path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")

        suffix = path.suffix.lower()
        if suffix not in self.SUPPORTED_EXTENSIONS:
            raise ValueError(f"Unsupported file type: {suffix}")

        logger.info(f"Loading file: {file_path}")
        docs = SimpleDirectoryReader(input_files=[file_path]).load_data()

        # For Excel/CSV, do row-level chunking
        if suffix in {".xlsx", ".xls", ".csv"}:
            docs = self._table_to_structured_text(docs, file_path)

        return docs

    def _table_to_structured_text(self, documents: list[Document], file_path: str) -> list[Document]:
        structured_docs = []
        path = Path(file_path)
        suffix = path.suffix.lower()

        for doc in documents:
            try:
                if suffix in [".xlsx", ".xls"]:
                    xls = pd.ExcelFile(file_path)
                    for sheet_name in xls.sheet_names:
                        df = pd.read_excel(xls, sheet_name=sheet_name)
                        structured_docs.extend(
                            self._rows_to_docs(df, file_name=path.name, sheet_name=sheet_name, file_type=suffix)
                        )
                elif suffix == ".csv":
                    df = pd.read_csv(file_path)
                    structured_docs.extend(
                        self._rows_to_docs(df, file_name=path.name, file_type=suffix)
                    )
                else:
                    structured_docs.append(doc)
            except Exception as e:
                logger.error(f"Error processing table {file_path}: {e}")
                structured_docs.append(doc)
        return structured_docs

    def _rows_to_docs(self, df: pd.DataFrame, file_name: str, sheet_name: str = None, file_type: str = None) -> list[Document]:
        docs = []
        headers = ", ".join(df.columns)
        for idx, row in df.iterrows():
            row_text = ", ".join([str(x) for x in row.tolist()])
            text = f"Headers: {headers}\nRow {idx + 1}: {row_text}"
            metadata = {
                "file_name": file_name,
                "sheet_name": sheet_name,
                "row_number": int(idx + 1),
                "file_type": file_type
            }
            docs.append(Document(text=text, metadata=metadata))
        return docs


class TextCleaner:
    def clean(self, text: str) -> str:
        return clean_text(text=text)


class Chunker:
    def __init__(self) -> None:
        settings = get_settings()
        self.chunk_size = settings.chunk_size_tokens
        self.chunk_overlap = settings.chunk_overlap_tokens

    def chunk_records(self, records: list[dict], doc_id: str, file_name: str) -> list[ChunkRecord]:
        chunks: list[ChunkRecord] = []
        chunk_counter = 0
        for rec in records:
            text = rec.get("text", "").strip()
            if not text:
                continue
            tokens = tokenize_words(text)
            for token_slice in overlap_chunks(tokens, self.chunk_size, self.chunk_overlap):
                chunk_text = " ".join(token_slice).strip()
                if not chunk_text:
                    continue
                chunk_counter += 1
                chunks.append(
                    ChunkRecord(
                        text=chunk_text,
                        doc_id=doc_id,
                        file_name=file_name,
                        file_type=rec.get("file_type", None),
                        page_number=int(rec.get("page_number", 0)),
                        chunk_id=f"{doc_id}-chunk-{chunk_counter}",
                        chunk_type=str(rec.get("chunk_type", "text"))
                    )
                )
        return chunks


class IngestionPipeline:
    def __init__(self):
        self.loader = DocumentLoader()
        self.cleaner = TextCleaner()
        self.chunker = Chunker()
        self.milvus = MilvusVectorStore()
        self.setting = get_settings()

    def process_file(self, file_path: str) -> List[ChunkRecord]:
        path = Path(file_path)
        file_name = path.name
        raw_docs = self.loader.load_data(file_path)

        normalized_records = []
        for doc in raw_docs:
            text = self.cleaner.clean(doc.text)
            if text:
                record = dict(doc.metadata)
                record["text"] = text
                normalized_records.append(record)

        # Generate doc_id based on filename + hash
        file_hash = hashlib.sha1(path.read_bytes()).hexdigest()[:12]
        doc_id = f"{path.stem}-{file_hash}"

        # Chunk records
        chunks = self.chunker.chunk_records(normalized_records, doc_id=doc_id, file_name=file_name)
        return chunks

    def ingest_to_milvus(self, chunks: List[ChunkRecord], batch_size: int = 1000) -> None:
        """
        Generate embeddings for chunks and insert into Milvus in batches.
        """
        total_chunks = len(chunks)
        logger.info(f"Starting ingestion of {total_chunks} chunks to Milvus")

        for i in range(0, total_chunks, batch_size):
            batch = chunks[i:i + batch_size]
            embeddings = []

            for chunk in batch:
                emb = ollama.embeddings(model=self.setting.embedding_model, prompt=chunk.text)
                print(emb)
                if len(emb) != 768:
                    raise ValueError(f"Embedding length mismatch for chunk {chunk.chunk_id}: {len(emb)} != 768")
                embeddings.append(emb)

            self.milvus.insert(batch, embeddings)
            logger.info(f"Inserted batch {i // batch_size + 1} ({len(batch)} chunks)")

        logger.info(f"Ingestion complete. Total entities: {self.milvus.collection.num_entities}")
        print("Total entities:", self.milvus.collection.num_entities)

In [ ]:
import logging 
import requests

import settings
from exceptions import OllamaTimeoutError



class OllamaClient:
    def __init__(self)->None:
        self.settings = settings.get_settings()
        self.base_url = self.settings.ollama_base_url.rstrip("/")
        self.timeout = self.settings.request_timeout_seconds
        self.logger = logging.getLogger(self.__class__.__name__)

    
    def embed(self, text: str)->list[float]:
        url = f"{self.base_url}/api/embeddings"
        payload = {"model": self.settings.embedding_model, "prompt": text}

        try:
            response = requests.post(url, json=payload, timeout=self.timeout)
            response.raise_for_status()
        except requests.Timeout as exc:
            raise OllamaTimeoutError("Embedding request timed out") from exc
        except requests.RequestException as exc:
            raise OllamaTimeoutError(f"Embedding request failed: {exc}") from exc
        

        data = response.json()
        embeddings = data.get("embedding")
        if not embeddings:
            raise OllamaTimeoutError("Ollama embedding API returned empty embedding")
        return embeddings
    
    
    def generate(self, messages: list[dict], temperature: float = 0.5)->str:
        url = f"{self.base_url}/api/chat"
        payload = {
            "model": self.settings.llm_model,
            "messages": messages,
            "stream": False,
            "options": {"temperature": temperature},
        }

        try:
            response = requests.post(url, json=payload, timeout=self.timeout)
            response.raise_for_status()
        except requests.Timeout as exc:
            raise OllamaTimeoutError("Generate request timeout") from exc
        except requests.RequestException as exc:
            raise OllamaTimeoutError(f"Generattion request failed: {exc}") from exc
        
        data = response.json()
        msg = data.get("message", {})
        content = msg.get("content", "")
        if not content:
            raise OllamaTimeoutError("Ollama chat API returned empty response")
        return content.strip()